# BBM ↔ external-platform harmonization — findings

Extends *Breakdowns in Realizing the Digital Extended Specimen*, from the paper's 131-record sample to the full BBM fungal collection, and adds automated cross-platform record resolution (using rule based and LLM matching)

**Harmonization relationships** (README): for a BBM specimen and its counterpart on a public platform —
- **bidirectional** — we cite their id *and* they cite our catalog number back
- **unidirectional** — only one side cites the other (either direction)
- **absent** — same specimen, cited nowhere in either direction

For the reproducible paper run, start with `python scripts/run_audit.py`; this writes an audit summary and manifest that the notebook can display. Interactive/network cells are still marked; run the fetch scripts first (`get_bbm_records.py`, `get_mo_records.py`) when refreshing source data.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "scripts" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
import link_audit as la
print("repo:", ROOT)

repo: /Users/wfrankel/Desktop/breakdowns_DES


## What this run actually fetched

Read from `config` + the CSVs on disk, so the numbers below reflect **this** run,
not a remembered one. The `.env` locality/collector filters apply to **BBM only**
(`get_bbm_records.py`); the external pulls are scoped by collection / dataset /
MO seed instead. If BBM filters are ON, the "34,856 / collection-wide" figures in
§1 no longer hold — the check at the bottom of the cell flags that.

In [2]:
import csv, datetime, config as cfg

def _info(name):
    p = cfg.DATA_DIR / name
    if not p.exists():
        return "— not fetched yet"
    with open(p, encoding="utf-8", newline="") as f:
        n = sum(1 for _ in csv.DictReader(f))
    ts = datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
    return f"{n:>7,} rows   (fetched {ts})"

fc, fl = cfg.FILTER_COLLECTORS or [], cfg.FILTER_LOCALITY or []
bbm_filtered = bool(fc or fl)

print("BBM filters :", "OFF — full fungal collection"
      if not bbm_filtered else f"ON — collectors={fc} locality={fl}")
print("MO seeds    :", f"user={cfg.MO_USER or '—'}  location={cfg.MO_LOCATION or '—'}")
print("data dir    :", cfg.DATA_DIR)
print()
for n in ["bbm_records.csv", "mo_records.csv", "mycoportal_records.csv",
          "gbif_records.csv", "genbank_records.csv"]:
    print(f"  {n:<24} {_info(n)}")

if bbm_filtered:
    print("\n\u26a0  BBM filters are ON \u2014 the \'34,856 / collection-wide\' "
          "figures in \u00a71 assume NO filters and no longer apply.")

BBM filters : OFF — full fungal collection
MO seeds    : user=2873  location=1679
data dir    : /Users/wfrankel/Desktop/breakdowns_DES/data

  bbm_records.csv           34,856 rows   (fetched 2026-09-04 15:57)
  mo_records.csv             5,866 rows   (fetched 2026-09-04 11:23)
  mycoportal_records.csv    34,946 rows   (fetched 2026-09-04 11:32)
  gbif_records.csv          34,878 rows   (fetched 2026-09-04 12:22)
  genbank_records.csv          213 rows   (fetched 2026-09-04 16:08)


## Reproducible audit manifest

`scripts/run_audit.py` is the stable paper workflow: offline, rule-based, no LLM, and no live API lookups by default. It regenerates the main report CSVs and writes the manifest shown below. Use this as the provenance record for paper numbers; use later interactive cells to inspect details.


In [3]:
import pandas as pd
summary_path = ROOT / "reports" / "audit_summary.csv"
manifest_path = ROOT / "reports" / "audit_manifest.json"
if summary_path.exists():
    audit_summary = pd.read_csv(summary_path)
    print(f"audit summary: {summary_path}")
    print(f"audit manifest: {manifest_path}")
    display(audit_summary)
else:
    print("Run `python scripts/run_audit.py` from the repo root to create audit_summary.csv and audit_manifest.json.")


audit summary: /Users/wfrankel/Desktop/breakdowns_DES/reports/audit_summary.csv
audit manifest: /Users/wfrankel/Desktop/breakdowns_DES/reports/audit_manifest.json


,metric,value
0,bbm_records,34856
1,dap_correct,261
2,dap_gold_links,355
3,dap_mode,rules
4,dap_precision_linked_pct,93.9
5,dap_recall_pct,73.5
6,dap_review_required,0
7,dap_unmatched,77
8,dap_wrong,17
9,dap_wrong_link_rate_pct,6.1


## 1. BBM → Mushroom Observer (lookup by stored id)

MO is an **independent, upstream** platform, some records posted there originally, and holds no copy of our GUID. The only link is the id we recorded (`MO # 82752`). We scan `bbm_records.csv` for those, look each up on MO, and check whether MO cites us back (a free-text `UBC F#` note).

In [4]:
# NETWORK: queries Mushroom Observer
mo = la.MushroomObserver()
res = la.audit(mo)                     # scan -> lookup -> classify
c = res["counts"]
on_mo = c["bidirectional"] + c["unidirectional"]
print(f"BBM records scanned     : {res['n_rows']}")
print(f"records citing an MO id  : {res['n_with_ref']}  ({100*res['n_with_ref']/res['n_rows']:.2f}%)")
print(f"distinct MO ids          : {len(res['ref_map'])}")
print(f"  resolve on MO          : {on_mo}")
print(f"    bidirectional        : {c['bidirectional']}")
print(f"    unidirectional UBC→MO: {c['unidirectional']}")
print(f"  dangling               : {c['dangling']}")

BBM records scanned     : 34856
records citing an MO id  : 21  (0.06%)
distinct MO ids          : 20
  resolve on MO          : 20
    bidirectional        : 17
    unidirectional UBC→MO: 3
  dangling               : 0


## 1a. Identifier integrity (category 02) — sub-cases

Category 02 is *not* "identifier missing" (that is 01). Per the paper (§5.1.3) it is a cross-reference that **is present but compromised**, in one of three ways:

- **wrong** — the id resolves to a *different* specimen (the MO record cites another UBC catalog number): a mis-assigned or typo'd id.
- **wrong-field** — the id sits in a free-text column, not a structured cross-reference field, so it does not propagate downstream.
- **hanging** — the id carries no recognizable prefix, so it is unreadable as a cross-reference without insider knowledge.

We now test these **per record** instead of tagging every independent-platform link. Detecting *wrong* is done per BBM citation, so it is still caught when a good record co-cites the same MO id (which grouping by MO id would otherwise mask). `hanging` is not separately detectable here — BBM stores every MO id with an `MO #` prefix, so the scanner only ever finds prefixed ids. `wrong-field` is reported (not asserted) until `MushroomObserver.reference_fields` names the Specify field that counts as a structured cross-reference; every MO id currently lives in `co_remarks`.

In [5]:
# Category 02 — identifier-integrity sub-cases in the MO audit (§5.1.3).
# Reuses `res` from the section-1 audit; re-run that cell first if `res` is undefined.
try:
    res
except NameError:
    res = la.audit(la.MushroomObserver())            # NETWORK

resolved = sum(1 for r in res["rows"] if r["exists"])
clean = sum(1 for r in res["rows"] if r["cites_us_back"])
c02 = res["cat02"]
print(f"resolved correspondences        : {resolved}")
print(f"  clean bidirectional           : {clean}")
print(f"  02 wrong id (UBC records)      : {c02['id_wrong']}")
print(f"  02 wrong-field (asserted)      : {c02['wrong_field']}")
for w in res["wrong"]:
    print(f"      BBM {w['bbm']} -> MO#{w['ref']}: MO cites {w['back_refs']}, not {w['our_ids']}")

cols = sorted({r["ref_source_cols"] for r in res["rows"] if r["ref_source_cols"]})
print(f"\ncolumns MO ids were scanned from : {cols}")
print("(all in a free-text notes field; set MushroomObserver.reference_fields to")
print(" assert wrong-field once the structured cross-reference field is confirmed.)")

resolved correspondences        : 20
  clean bidirectional           : 17
  02 wrong id (UBC records)      : 2
  02 wrong-field (asserted)      : 0
      BBM F023000 -> MO#66139: MO cites F23003, not 16D88EF3-EB8E-AC44-8221-1D11EE7085FA; F023000; F23000
      BBM F023033 -> MO#82705: MO cites F23090, not D02F4482-431E-7E4F-917C-D3087547ABF0; F023033; F23033

columns MO ids were scanned from : ['co_remarks']
(all in a free-text notes field; set MushroomObserver.reference_fields to
 assert wrong-field once the structured cross-reference field is confirmed.)


## 2. MyCoPortal (harvested — matched by GUID)

MyCoPortal is the **opposite coupling**: it is **harvested wholesale from our Specify database** via Symbiota, so every MP record carries our GUID (`occurrenceID`) and F-number (`catalogNumber`) *by propagation*. The `Mycoportal # UBC#####` strings in our records are **legacy free-text, not queryable ids** (they even ride along verbatim into MP's `occurrenceRemarks`). The reliable link is the **GUID**.

Confirmed against the live Symbiota API (`/api/v2/occurrence?occurrenceID=<guid>`):
- UBC fungi on MyCoPortal (`collid 49`): **34,946 records** ≈ our 34,856 — essentially the whole collection.
- match: MP `occurrenceID` == BBM `guid`; MP `catalogNumber` == BBM `F#`.

**Coupling contrast, quantified:** loosely-coupled MO → **0.06 %** cross-referenced; tightly-coupled MyCoPortal → **~complete**. That is the paper's central argument in two numbers. (A GUID-discovery audit that counts MP presence + harvest gaps per record is the natural next script.)

In [6]:
# OFFLINE: how many records carry the legacy 'Mycoportal #' annotation
mp = la.MyCoPortal()
ref_map, n_rows, n_with_ref = la.scan(mp, str(la.INPUT))
print(f"records with a legacy 'Mycoportal #' note : {n_with_ref} (of {n_rows})")
print("→ not a queryable id; real MP linkage is the GUID (see above)")

records with a legacy 'Mycoportal #' note : 75 (of 34856)
→ not a queryable id; real MP linkage is the GUID (see above)


## 3. Ceska / Observatory Hill quadrants — cross-platform resolution (README §2)

To answer *how many MO records are ours but unconnected* we resolve records **by attributes**, using the evaluator subsystem **vendored** into `scripts/evaluators/` (copied from the orchestration framework) — `RuleBasedEvaluator` + `LLMEvaluator` — via `resolve.py`. Both platforms are shaped into orchestration's record contract, blocked by genus, and clustered; a cluster with a BBM and an MO record is a match. Each match is then scored into a quadrant by crossing the attribute match with the recorded cross-references.

**Seeds:** MO user 2873 (Ceska) = 5,866 obs; MO location 1679 (Observatory Hill) = 2,707 obs.

> Requires `bbm_records.csv` (joined) and `mo_records.csv`. The default cell below is rule-based only. The optional LLM cell samples a capped DAP-unmatched review subset, not the full MO corpus.

In [7]:
import resolve as R, platforms as P
from collections import Counter
mo = P.PLATFORMS["mo"]
bbm_p, mo_p = R.DATA_DIR / "bbm_records.csv", R.DATA_DIR / "mo_records.csv"

# Default notebook runs are rule-based only. The LLM tier is experimental,
# bounded, and guardrailed; run the optional cell below for it.
USE_LLM = False

if bbm_p.exists() and mo_p.exists():
    bbm_rows, bmeta = R.load_bbm(str(bbm_p), mo)
    plat_rows, pmeta = R.load_platform(str(mo_p))
    meta = {**bmeta, **pmeta}
    pairs, dups = R.resolve(bbm_rows, plat_rows, meta, use_llm=USE_LLM)
    q = Counter(R.quadrant(b, m, meta) for b, m, _ in pairs)
    how = Counter(h for _, _, h in pairs)
    print(f"LLM tier enabled       : {USE_LLM}")
    print(f"BBM records            : {len(bbm_rows)}")
    print(f"MO Ceska/OH records    : {len(plat_rows)}")
    print(f"cross-platform matches : {len(pairs)}   (by method: {dict(how)})")
    for k in ("bidirectional","unidirectional_ubc_to_platform","unidirectional_platform_to_ubc","absent"):
        print(f"  {k:34} {q.get(k,0)}")
else:
    print("Run get_bbm_records.py and get_mo_records.py first, then re-run this cell.")

LLM tier enabled       : False
BBM records            : 34856
MO Ceska/OH records    : 5866
cross-platform matches : 3401   (by method: {'similar': 2008, 'strict': 1393})
  bidirectional                      16
  unidirectional_ubc_to_platform     2
  unidirectional_platform_to_ubc     726
  absent                             2657


In [8]:
# Optional LLM review experiment. Leave off for normal Run All.
# This samples DAP records the rule baseline missed; it does not run LLM over the full corpus.
import importlib, os
import pandas as pd
import validate_dap as V; importlib.reload(V)

RUN_LLM_EXPERIMENT = True
LLM_REVIEW_LIMIT = 10
LLM_REVIEW_REASON = "name_below_rule_threshold"

if RUN_LLM_EXPERIMENT:
    if not os.getenv("LLM_MODEL"):
        raise RuntimeError("RUN_LLM_EXPERIMENT=True needs LLM_MODEL set in .env.")
    rules_unmatched = V.REPORTS_DIR / "dap_validation_rules_unmatched_reasons.csv"
    if not rules_unmatched.exists():
        V.validate(use_llm=False, per_genus=150, bbm_path=str(V.DATA_DIR / "bbm_records.csv"))
    pool = pd.read_csv(rules_unmatched)
    pool = pool[pool["diagnostic_reason"] == LLM_REVIEW_REASON].head(LLM_REVIEW_LIMIT)
    review_ids = pool["mo_id"].astype(str).tolist()
    print(f"LLM review subset: {len(review_ids)} records ({LLM_REVIEW_REASON})")
    if review_ids:
        llm_summary = V.validate(
            use_llm=True,
            per_genus=40,
            bbm_path=str(V.DATA_DIR / "bbm_records.csv"),
            only_mo_ids=review_ids,
            label="rules+llm_review",
        )
        display(pd.DataFrame([llm_summary]))
        display(pd.read_csv(V.REPORTS_DIR / "dap_validation_rules+llm_review.csv"))
    else:
        print("No rows matched the requested LLM_REVIEW_REASON; no LLM calls made.")
else:
    print("LLM review experiment skipped. Set RUN_LLM_EXPERIMENT = True to run a capped DAP-unmatched subset.")


LLM review subset: 10 records (name_below_rule_threshold)


,mode,slug,gold_links,present_in_corpus,linked,correct,wrong,unmatched_present,not_in_corpus,recall_all,...,precision_linked,wrong_link_rate,accuracy_on_gold,strict_correct,similar_correct,llm_correct,review_required_links,output,wrong_links_output,unmatched_output
0,rule+LLM-leftover subset,rules+llm_review,10,10,0,0,0,10,0,0.0,...,0,0,0.0,0,0,0,0,/Users/wfrankel/Desktop/breakdowns_DES/reports...,/Users/wfrankel/Desktop/breakdowns_DES/reports...,/Users/wfrankel/Desktop/breakdowns_DES/reports...


,mo_id,gold_F,matched_F,tier,result,diagnostic_reason,diagnostic_detail,review_required,accepted,llm_reason,...,matched_bbm_date,matched_bbm_locality,mo_record_id,mo_platform,mo_sci_name,mo_genus,mo_species,mo_collector,mo_date,mo_locality
0,166513,F19584,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.654,NaN,NaN,NaN,...,NaN,NaN,MO:166513,mo,cortinarius decipientoides,cortinarius,decipientoides,oluna & adolf ceska,2008-11-05,"observatory hill, victoria, british columbia, ..."
1,210193,F19553,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.474,NaN,NaN,NaN,...,NaN,NaN,MO:210193,mo,inocybe ceskae,inocybe,ceskae,oluna & adolf ceska,2008-10-26,"observatory hill, victoria, british columbia, ..."
2,209961,F19266,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.5,NaN,NaN,NaN,...,NaN,NaN,MO:209961,mo,inocybe ceskae,inocybe,ceskae,oluna & adolf ceska,2004-12-11,"observatory hill, victoria, british columbia, ..."
3,202610,F19645,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.5,NaN,NaN,NaN,...,NaN,NaN,MO:202610,mo,inocybe ericetorum,inocybe,ericetorum,oluna & adolf ceska,2009-05-14,"observatory hill, victoria, british columbia, ..."
4,166825,F19593,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.667,NaN,NaN,NaN,...,NaN,NaN,MO:166825,mo,cortinarius cinnabarinus,cortinarius,cinnabarinus,oluna & adolf ceska,2008-11-21,"observatory hill, victoria, british columbia, ..."
5,132649,F16319,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.65,NaN,NaN,NaN,...,NaN,NaN,MO:132649,mo,cortinarius miwok,cortinarius,miwok,oluna & adolf ceska,2008-02-28,"observatory hill, victoria, british columbia, ..."
6,72889,F19522,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.579,NaN,NaN,NaN,...,NaN,NaN,MO:72889,mo,inocybe calida,inocybe,calida,oluna & adolf ceska,2008-07-03,"observatory hill, victoria, british columbia, ..."
7,71652,F19441,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.579,NaN,NaN,NaN,...,NaN,NaN,MO:71652,mo,inocybe glabrodisca,inocybe,glabrodisca,oluna & adolf ceska,2007-07-22,"observatory hill, victoria, british columbia, ..."
8,146033,F33821,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.48,NaN,NaN,NaN,...,NaN,NaN,MO:146033,mo,paxillus involutus,paxillus,involutus,oluna & adolf ceska,2013-09-17,"observatory hill, victoria, british columbia, ..."
9,64347,F24557,NaN,NaN,absent,name_below_rule_threshold,Best gold name similarity is 0.692,NaN,NaN,NaN,...,NaN,NaN,MO:64347,mo,butyriboletus persolidus,butyriboletus,persolidus,oluna & adolf ceska,2010-09-28,"observatory hill, victoria, british columbia, ..."


## 3a. Duplicate records (category 06) — attribute-level, within a platform

`resolve()` now also returns *same-platform* duplicate pairs: two records in one
attribute-matched cluster that share a platform (two MO observations, or two BBM
catalog entries, for one specimen). This is the **independent-platform side of
category 06** — the harvested side (duplicate GUIDs) is covered by
`guid_discovery.py` in §8 (`present_dup`). These are **candidates for review**,
not confirmed duplicates: attribute matching alone (name + date + locality) is
weak evidence, so a curator confirms before any merge. Written to
`reports/mo_duplicates.csv` by `resolve.py`.

In [9]:
# Category 06 - attribute-level duplicate records (same platform, same cluster).
# Report CLUSTERS, not raw pairs: a cluster of k records yields k*(k-1)/2 pairs,
# so pairs overstate. These are candidates (weak attribute evidence) for review.
from collections import Counter, defaultdict
if 'dups' in dir():
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        parent[find(a)] = find(b)
    for a, b, pl, how in dups:
        union(a, b)
    clusters = defaultdict(set)
    for a, b, pl, how in dups:
        clusters[find(a)] |= {a, b}
    sizes = sorted((len(s) for s in clusters.values()), reverse=True)
    by_plat = Counter(pl for _, _, pl, _ in dups)
    by_how  = Counter(how for _, _, _, how in dups)
    print(f'duplicate pairs (cat 06)   : {len(dups)}   by platform {dict(by_plat)}')
    print(f'duplicate clusters         : {len(sizes)}   (size-2: {sum(1 for s in sizes if s==2)}, >=5: {sum(1 for s in sizes if s>=5)})')
    print(f'largest clusters           : {sizes[:8]}')
    print(f'match tier                 : {dict(by_how)}   # strict = strong, similar = weak/candidate')
    print('NOTE: on the single-collector/single-locality Ceska-OH corpus, similar-tier')
    print('      clusters are dominated by distinct specimens sharing taxon+site.')
    print('      Confirm via identifier evidence, DAP validation, or curator review before trusting counts.')
else:
    print('Run cell 3 (§3) first.')

duplicate pairs (cat 06)   : 22941   by platform {'BBM': 19269, 'mo': 3672}
duplicate clusters         : 4927   (size-2: 3322, >=5: 439)
largest clusters           : [52, 46, 36, 28, 27, 26, 24, 22]
match tier                 : {'similar': 19042, 'strict': 3899}   # strict = strong, similar = weak/candidate
NOTE: on the single-collector/single-locality Ceska-OH corpus, similar-tier
      clusters are dominated by distinct specimens sharing taxon+site.
      Confirm via identifier evidence, DAP validation, or curator review before trusting counts.


## 4. Figure 2 comparison

In [10]:
import pandas as pd
paper = {"UBC records citing MO":19, "bidirectional":8,
         "unidirectional (UBC→MO)":10, "dangling / wrong id":1, "bidirectional rate":"42%"}
ours  = {"UBC records citing MO":len(res["ref_map"]), "bidirectional":c["bidirectional"],
         "unidirectional (UBC→MO)":c["unidirectional"], "dangling / wrong id":c["dangling"],
         "bidirectional rate":f"{100*c['bidirectional']/max(len(res['ref_map']),1):.0f}%"}
pd.DataFrame({"Kholmatova 2026 (Fig 2, Phase I)":paper, "This audit (full collection)":ours})

,"Kholmatova 2026 (Fig 2, Phase I)",This audit (full collection)
UBC records citing MO,19,20
bidirectional,8,17
unidirectional (UBC→MO),10,3
dangling / wrong id,1,0
bidirectional rate,42%,85%


## 5. GBIF (harvested — matched by GUID)

GBIF is the **same coupling as MyCoPortal**: our Specify collection is **harvested wholesale** into GBIF as a published dataset (`datasetKey ca1bcd7e-7387-42f9-81ba-1470db55e3e8`), so every GBIF occurrence carries our GUID (`occurrenceID`) and F-number (`catalogNumber`) *by propagation*. Our records almost never store a GBIF id in free text — the reliable link is again the **GUID**, discovered from GBIF's side, not cited from ours.

- **Reference direction (below):** BBM free text → GBIF id. Expected ~0 — GBIF ids are not something a curator types into a specimen record.
- **Discovery direction (the real one):** `python scripts/get_records.py --platform gbif` pulls the whole dataset by `datasetKey` (~35k occurrences) and matches each back to a BBM record by `occurrenceID == guid`. Like MyCoPortal, presence should be **~complete**, so the finding is the *harvest gap* — our records with no GBIF twin — not missing cross-references.

In [11]:
# OFFLINE: GBIF ids appearing in BBM free text (real link is the GUID, discovered from GBIF)
gbif = la.PLATFORMS["gbif"]
ref_map, n_rows, n_with_ref = la.scan(gbif, str(la.INPUT))
print(f"BBM records scanned                 : {n_rows}")
print(f"records citing a GBIF id in free text: {n_with_ref}  (harvested platform → expected ~0)")
print(f"coupling                            : {gbif.coupling}  (matched by GUID / occurrenceID)")
print(f"GBIF dataset key                    : {gbif.DATASET_KEY}")
print("→ discovery direction: python scripts/get_records.py --platform gbif  (~35k occurrences)")

BBM records scanned                 : 34856
records citing a GBIF id in free text: 0  (harvested platform → expected ~0)
coupling                            : harvested  (matched by GUID / occurrenceID)
GBIF dataset key                    : ca1bcd7e-7387-42f9-81ba-1470db55e3e8
→ discovery direction: python scripts/get_records.py --platform gbif  (~35k occurrences)


## 6. GenBank (independent — matched by stored accession)

GenBank is coupled like **Mushroom Observer**: an **independent upstream** database where the link exists only if *our* record stores the sequence **accession** (`GenBank KX691234`), and it is *bidirectional* only if the GenBank record's own definition/notes cite our `UBC F#` back. Nothing is harvested, so — exactly as the paper predicts for loosely-coupled platforms — coverage is **sparse**: only sequenced specimens have an accession at all.

- **Reference direction (below):** BBM free text → GenBank accession, then look each up via NCBI eutils and check for a `UBC F#` back-reference. This is cheap: only records that actually cite an accession trigger a lookup.
- **Discovery direction:** `python scripts/get_records.py --platform genbank` searches NCBI for `UBC` / `University of British Columbia` vouchers — the way to find sequences that exist but were never linked from our side.

In [12]:
# NETWORK: only BBM records that cite a GenBank accession are looked up (expected sparse)
gb = la.PLATFORMS["genbank"]
res = la.audit(gb)
c = res["counts"]
print(f"BBM records scanned              : {res['n_rows']}")
print(f"records citing a GenBank accession: {res['n_with_ref']}")
print(f"distinct accessions               : {res['n_ids']}")
print(f"  bidirectional (cites UBC back)  : {c['bidirectional']}")
print(f"  unidirectional (UBC→GenBank)    : {c['unidirectional']}")
print(f"  dangling (accession not found)  : {c['dangling']}")
print("→ discovery direction: python scripts/get_records.py --platform genbank")

BBM records scanned              : 34856
records citing a GenBank accession: 336
distinct accessions               : 331
  bidirectional (cites UBC back)  : 2
  unidirectional (UBC→GenBank)    : 248
  dangling (accession not found)  : 81
→ discovery direction: python scripts/get_records.py --platform genbank


## 7. Cross-platform representation — explicit-link summary (Goal 1 / paper Fig 3)

This table is the Goal 1 / paper-facing representation table. It counts **explicit identifier links** in the latest fetched CSVs, not resolver candidates.

Use this table when answering: "how many records explicitly cite each other across platforms?" For Mushroom Observer, that means BBM text fields containing an MO id and MO records carrying a UBC `F#`. These are the closest automated equivalent of the paper's bidirectional / unidirectional / incorrect harmonization assessment.

Do **not** compare these counts one-to-one with `mo_resolution.csv`. The resolver table answers a different question: "among BBM/MO candidate pairs inferred from name/date/locality/collector, what cross-reference quadrant does each candidate pair fall into?" That table is useful for Goal 2 and review queues, but the explicit-link table below is the Goal 1 headline.


In [ ]:
# Goal 1 explicit-link representation table. OFFLINE — reads the latest lineage/audit outputs.
import pandas as pd
from config import DATA_DIR, REPORTS_DIR


def csv_len(path):
    return len(pd.read_csv(path, dtype=str, keep_default_na=False))

lineage = pd.read_csv(REPORTS_DIR / "specimen_lineage_report.csv", dtype=str, keep_default_na=False)
bbm_lineage = lineage[~lineage["specimen_key"].str.startswith("ORPHAN:")]
orphan_lineage = lineage[lineage["specimen_key"].str.startswith("ORPHAN:")]
mo_counts = bbm_lineage["mo_explicit_link_status"].value_counts()

mp_counts = bbm_lineage["mycoportal_status"].value_counts()
gbif_counts = bbm_lineage["gbif_status"].value_counts()
orphan_sources = orphan_lineage["specimen_key"].str.extract(r"^ORPHAN:([^:]+):")[0].value_counts()

genbank = pd.read_csv(REPORTS_DIR / "genbank_linkage.csv", dtype=str, keep_default_na=False)
gb_ubc = genbank["ubc_cites_accession"].str.lower().eq("true")
gb_voucher = genbank["voucher_cites_F"].str.lower().eq("true")

rows = [
    {
        "platform": "mo",
        "metric basis": "explicit MO id / UBC F# cross-reference, BBM-row unit",
        "coupling": "independent",
        "records": csv_len(DATA_DIR / "mo_records.csv"),
        "bidirectional": int(mo_counts.get("bidirectional", 0)),
        "uni UBC->plat": int(mo_counts.get("unidirectional_ubc_to_mo", 0)),
        "uni plat->UBC · 01": int(mo_counts.get("unidirectional_mo_to_ubc", 0)),
        "wrong id · 02": int(mo_counts.get("wrong_id", 0)),
    },
    {
        "platform": "mycoportal",
        "metric basis": "explicit GUID harvest join",
        "coupling": "harvested",
        "records": csv_len(DATA_DIR / "mycoportal_records.csv"),
        "present (harvested)": int(mp_counts.get("present", 0)),
        "harvest gap · 03": int(mp_counts.get("harvest_gap", 0)),
        "orphan · 03": int(orphan_sources.get("mycoportal", 0)),
        "duplicate · 06": int(mp_counts.get("present_dup", 0)),
    },
    {
        "platform": "gbif",
        "metric basis": "explicit GUID harvest join",
        "coupling": "harvested",
        "records": csv_len(DATA_DIR / "gbif_records.csv"),
        "present (harvested)": int(gbif_counts.get("present", 0)),
        "harvest gap · 03": int(gbif_counts.get("harvest_gap", 0)),
        "orphan · 03": int(orphan_sources.get("gbif", 0)),
        "duplicate · 06": int(gbif_counts.get("present_dup", 0)),
    },
    {
        "platform": "genbank",
        "metric basis": "DAP accession ground truth + explicit accession/voucher refs",
        "coupling": "independent",
        "records": len(genbank),
        "bidirectional": int((gb_ubc & gb_voucher).sum()),
        "uni UBC->plat": int((gb_ubc & ~gb_voucher).sum()),
        "uni plat->UBC · 01": int((~gb_ubc & gb_voucher).sum()),
        "absent explicit x-ref · 01": int((~gb_ubc & ~gb_voucher).sum()),
        "UBC missing accession · 01": int((~gb_ubc).sum()),
    },
]

order = ["platform", "metric basis", "coupling", "records", "present (harvested)",
         "harvest gap · 03", "orphan · 03", "duplicate · 06", "bidirectional",
         "uni UBC->plat", "uni plat->UBC · 01", "wrong id · 02",
         "absent explicit x-ref · 01", "UBC missing accession · 01"]
table = pd.DataFrame(rows).reindex(columns=order).fillna("—")
table


## 8. Absence from repositories (category 03) — GUID reconciliation

Harvested platforms carry our GUID, so coverage is a clean GUID join (`guid_discovery.py`, offline). **Harvest gap** = a BBM specimen absent from the platform (never published downstream). **Orphan** = a platform record whose GUID isn't in BBM (investigate). Both are category 03. The undigitized-backlog sub-case of 03 is *not* measurable here — a backlog specimen has no digital trace to join on.

In [14]:
# Category 03 — harvest-gap / orphan coverage per harvested platform. OFFLINE.
import guid_discovery as gd
import platforms as P
import pandas as pd
from config import DATA_DIR

out = []
for name, label in [("mycoportal", "MyCoPortal"), ("gbif", "GBIF")]:
    disc = DATA_DIR / f"{name}_records.csv"
    if not disc.exists():
        continue
    res = gd.audit(P.PLATFORMS[name], str(DATA_DIR / "bbm_records.csv"), str(disc))
    c = res["counts"]
    present = c["present"] + c["present_dup"]
    out.append({"platform": label, "BBM rows": res["n_bbm"], "present": present,
                "coverage %": round(100 * present / res["n_bbm"], 1),
                "harvest gap \u00b7 03": c["harvest_gap"],
                "duplicate \u00b7 06": c["present_dup"],
                "orphan \u00b7 03": res["n_orphan"]})
pd.DataFrame(out)

,platform,BBM rows,present,coverage %,harvest gap · 03,duplicate · 06,orphan · 03
0,MyCoPortal,34856,34633,99.4,223,0,313
1,GBIF,34856,33099,95.0,1757,0,1779


## 9. Validation against the 2025 DAP ground truth (paper contribution C2)

DAP audit hand-linked Mushroom Observer records to UBC `F#`, GenBank accession, and the cross-reference each still needed. This section validates the MO→UBC matching pipeline against that ground truth and reports recall, precision among linked records, wrong-F# rate, and how many links require review.

**Current rule-based baseline:** 261/355 = **73.5%** recovered, split 237 `strict` / 24 `similar`, with 17 wrong-F# links and 77 unmatched records.

The LLM tier is experimental. It is off by default below because local Ollama calls can time out and because LLM-derived links are treated as `review_required`, not accepted paper results, until DAP validation shows that recall improves without an unacceptable wrong-link rate.

*Caveats:* DAP is a 2025 snapshot of what still needed fixing, so MO<->F# **matching** is stable ground truth but the "add MO #" direction labels may be partly resolved in the 2026 BBM refetch. `per_genus` caps decoys: recall is exact, while wrong-link count is measured against the bounded decoy pool. Duplicate-record accuracy is not yet measured here because DAP supplies MO→UBC links, not confirmed duplicate/non-duplicate labels; duplicate output remains candidate counts until that gold set exists.

In [15]:
# C2 validation. Default run is rule-based only; LLM modes are optional.
import logging, importlib, os
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
import validate_dap as V; importlib.reload(V)
BBM = str(V.DATA_DIR / 'bbm_records.csv')

RUN_LLM_VALIDATION = False
summaries = [V.validate(use_llm=False, per_genus=150, bbm_path=BBM)]

if RUN_LLM_VALIDATION and not os.getenv("LLM_MODEL"):
    raise RuntimeError("RUN_LLM_VALIDATION=True needs LLM_MODEL set in .env.")

if RUN_LLM_VALIDATION:
    summaries.append(V.validate(use_llm=True, per_genus=150, bbm_path=BBM))
    summaries.append(V.validate(use_llm=False, per_genus=40, bbm_path=BBM, force_llm=True))

summary = pd.DataFrame(summaries)
for col in ["recall_all", "recall_present", "precision_linked", "wrong_link_rate", "accuracy_on_gold"]:
    summary[col] = (summary[col] * 100).round(1)
summary.to_csv(V.REPORTS_DIR / "dap_validation_summary.csv", index=False)
summary[["mode", "gold_links", "linked", "correct", "wrong", "unmatched_present",
         "recall_all", "precision_linked", "wrong_link_rate",
         "strict_correct", "similar_correct", "llm_correct",
         "review_required_links", "output"]]


GT gold matches: 355 | present in mo_records.csv: 355 | BBM in-genera: 7862
DAP matching validation (Observatory Hill, rule-based)
  gold MO->F# links            : 355
  recovered correctly          : 261  (73.5% of all, 73.5% of present)
    strict / similar / llm     : 237 / 24 / 0
  linked to WRONG F#           : 17
  precision among linked       : 93.9%
  wrong-link rate among linked : 6.1%
  review-required links        : 0
  gold present but unmatched   : 77
  gold MO id not in corpus     : 0
Saved per-record → /Users/wfrankel/Desktop/breakdowns_DES/reports/dap_validation_rules.csv
Saved wrong-link diagnostics → /Users/wfrankel/Desktop/breakdowns_DES/reports/dap_validation_rules_wrong_links.csv
Saved unmatched diagnostics → /Users/wfrankel/Desktop/breakdowns_DES/reports/dap_validation_rules_unmatched_reasons.csv


,mode,gold_links,linked,correct,wrong,unmatched_present,recall_all,precision_linked,wrong_link_rate,strict_correct,similar_correct,llm_correct,review_required_links,output
0,rule-based,355,278,261,17,77,73.5,93.9,6.1,237,24,0,0,/Users/wfrankel/Desktop/breakdowns_DES/reports...


### Latest validation files found

This cell summarizes any full DAP validation outputs already present in `reports/`, including CLI runs. Use this to compare `rules`, `rules+llm`, and `llm` results after running validation outside the notebook. Small review/subset files are excluded from the paper-facing comparison.


In [ ]:
# Summarize existing full DAP validation files, including CLI-generated LLM runs.
from pathlib import Path
from collections import Counter
import csv
import pandas as pd

validation_rows = []
for path in sorted((V.REPORTS_DIR).glob("dap_validation_*.csv")):
    name = path.name
    if any(skip in name for skip in ["summary", "wrong_links", "unmatched_reasons", "reason_counts", "review", "subset"]):
        continue
    rows = list(csv.DictReader(open(path, newline="", encoding="utf-8")))
    if not rows or "result" not in rows[0]:
        continue
    counts = Counter(r["result"] for r in rows)
    linked = counts["correct"] + counts["wrong_F"]
    tiers = Counter(r["tier"] for r in rows if r["result"] == "correct")
    validation_rows.append({
        "file": name,
        "gold_links": len(rows),
        "correct": counts["correct"],
        "wrong": counts["wrong_F"],
        "unmatched": counts["absent"] + counts["not_in_corpus"],
        "recall %": round(100 * counts["correct"] / len(rows), 1),
        "precision linked %": round(100 * counts["correct"] / linked, 1) if linked else 0,
        "wrong-link rate %": round(100 * counts["wrong_F"] / linked, 1) if linked else 0,
        "strict correct": tiers["strict"],
        "similar correct": tiers["similar"],
        "llm correct": tiers["llm"],
        "review required": sum(str(r.get("review_required")).lower() == "true" for r in rows),
        "modified": pd.Timestamp(path.stat().st_mtime, unit="s"),
    })
validation_found = pd.DataFrame(validation_rows).sort_values("file")
display(validation_found)


### DAP diagnostics: wrong links and unmatched reasons

The validation CSV is also the improvement ledger. Wrong links compare the DAP gold BBM record, the BBM record the resolver chose, and the MO record side by side. Unmatched records are grouped by the first deterministic reason the gold pair failed the current matcher. After changing rules or the LLM scaffold, re-run the validation cell above and this cell to see which failure classes moved.

In [16]:
# Diagnostic tables for matcher/LLM improvement tracking.
from pathlib import Path

diag_slug = summary.iloc[0]["slug"] if "summary" in globals() and len(summary) else "rules"
wrong_path = V.REPORTS_DIR / f"dap_validation_{diag_slug}_wrong_links.csv"
unmatched_path = V.REPORTS_DIR / f"dap_validation_{diag_slug}_unmatched_reasons.csv"

wrong_links = pd.read_csv(wrong_path) if Path(wrong_path).exists() else pd.DataFrame()
unmatched = pd.read_csv(unmatched_path) if Path(unmatched_path).exists() else pd.DataFrame()

if unmatched.empty:
    print(f"No unmatched diagnostics found at {unmatched_path}")
else:
    reason_counts = (unmatched["diagnostic_reason"]
        .value_counts()
        .rename_axis("diagnostic_reason")
        .reset_index(name="n"))
    reason_counts.to_csv(V.REPORTS_DIR / f"dap_validation_{diag_slug}_unmatched_reason_counts.csv", index=False)
    display(reason_counts)

if wrong_links.empty:
    print(f"No wrong-link diagnostics found at {wrong_path}")
else:
    wrong_cols = ["mo_id", "gold_F", "matched_F", "tier", "diagnostic_reason", "diagnostic_detail",
                  "gold_bbm_sci_name", "gold_bbm_date", "gold_bbm_locality",
                  "matched_bbm_sci_name", "matched_bbm_date", "matched_bbm_locality",
                  "mo_sci_name", "mo_date", "mo_locality", "candidate_group_ids"]
    display(wrong_links[wrong_cols].head(20))

if not unmatched.empty:
    unmatched_cols = ["mo_id", "gold_F", "diagnostic_reason", "diagnostic_detail",
                      "gold_bbm_sci_name", "gold_bbm_genus", "gold_bbm_date", "gold_bbm_locality",
                      "mo_sci_name", "mo_genus", "mo_date", "mo_locality"]
    display(unmatched[unmatched_cols].head(30))


,diagnostic_reason,n
0,genus_mismatch_blocks_match,49
1,name_below_rule_threshold,25
2,date_conflict_exact,3


,mo_id,gold_F,matched_F,tier,diagnostic_reason,diagnostic_detail,gold_bbm_sci_name,gold_bbm_date,gold_bbm_locality,matched_bbm_sci_name,matched_bbm_date,matched_bbm_locality,mo_sci_name,mo_date,mo_locality,candidate_group_ids
0,167151,F19611,F19599,strict,ambiguous_same_name_locality,matched_score=11; gold_score=8; matched_date=2...,inocybe pseudodestricta,2008-11-25,"victoria, saanich, observatory hill",inocybe pseudodestricta,2008-11-22,"victoria, saanich, observatory hill",inocybe pseudodestricta,2008-11-22,"observatory hill, victoria, british columbia, ...",BBM:F019599; MO:167151
1,166830,F19599,F19611,similar,ambiguous_same_name_locality,matched_score=8; gold_score=8; matched_date=20...,inocybe pseudodestricta,2008-11-22,"victoria, saanich, observatory hill",inocybe pseudodestricta,2008-11-25,"victoria, saanich, observatory hill",inocybe pseudodestricta,2008-11-21,"observatory hill, victoria, british columbia, ...",BBM:F019611; MO:166830
2,159694,F25462,F24916,strict,decoy_scores_above_gold,matched_score=11; gold_score=7; matched_date=2...,botryobasidium conspersum,2012-04-28,"victoria, saanich peninsula, observatory hill;...",botryobasidium conspersum,2011-02-10,"victoria, saanich peninsula, observatory hill;...",botryobasidium conspersum,2011-02-10,"observatory hill, victoria, british columbia, ...",BBM:F024916; MO:159694
3,151558,F26101,F25839,similar,decoy_scores_above_gold,matched_score=8; gold_score=6; matched_date=20...,clitopilus aureicystidiatus,2013-10-08,"victoria, saanich peninsula, observatory hill;...",rhodocybe aureicystidiata,2013-02-20,"victoria, saanich peninsula, observatory hill;...",rhodocybe aureicystidiata,2013-10-08,"observatory hill, victoria, british columbia, ...",BBM:F025839; BBM:F026303; MO:151558
4,135031,F25934,F35035,similar,decoy_scores_above_gold,matched_score=7; gold_score=6; matched_date=; ...,ceriporiopsis subvermispora,2013-05-20,"victoria, saanich peninsula, observatory hill;...",gelatoporia subvermispora,NaN,vancouver island; saanich; observatory hill; b...,gelatoporia subvermispora,2013-05-20,"observatory hill, victoria, british columbia, ...",BBM:F035035; MO:135031
5,87641,F25391,F35157,similar,decoy_scores_above_gold,matched_score=7; gold_score=6; matched_date=; ...,simocybe haustellaris,2012-01-27,"victoria, saanich peninsula, observatory hill;...",pleuroflammula ragazziana,NaN,vancouver island; saanich; observatory hill; b...,pleuroflammula ragazziana,2012-01-27,"observatory hill, victoria, british columbia, ...",BBM:F035157; MO:87641
6,73091,F25079,F29330,strict,decoy_scores_above_gold,matched_score=11; gold_score=2; matched_date=2...,acanthophysellum lividocoeruleum,2011-05-28,"victoria, saanich peninsula, observatory hill;...",agaricus arvensis,2007-09-30,"victoria, saanich peninsula, observatory hill;...",agaricus arvensis,2007-09-30,"observatory hill, victoria, british columbia, ...",BBM:F029330; MO:73091
7,73025,F25079,F29449,strict,decoy_scores_above_gold,matched_score=11; gold_score=2; matched_date=2...,acanthophysellum lividocoeruleum,2011-05-28,"victoria, saanich peninsula, observatory hill;...",agaricus diminutivus,2007-11-07,"victoria, saanich peninsula, observatory hill;...",agaricus diminutivus,2007-11-07,"observatory hill, victoria, british columbia, ...",BBM:F029449; MO:73025
8,72975,F25079,F24235,similar,decoy_scores_above_gold,matched_score=8; gold_score=2; matched_date=20...,acanthophysellum lividocoeruleum,2011-05-28,"victoria, saanich peninsula, observatory hill;...",steccherinum ochraceum,2010-02-14,"victoria, saanich peninsula, observatory hill;...",steccherinum ochraceum,2010-03-05,"observatory hill, victoria, british columbia, ...",BBM:F024235; BBM:F024281; BBM:F024323; BBM:F02...
9,71514,F25079,F25123,strict,decoy_scores_above_gold,matched_score=11; gold_score=3; matched_date=2...,acanthophysellum lividocoeruleum,2011-05-28,"victoria, saanich peninsula, observatory hill;...",pluteus thomsonii,2011-06-30,"victoria, saanich peninsula, observatory hill;..

,mo_id,gold_F,diagnostic_reason,diagnostic_detail,gold_bbm_sci_name,gold_bbm_genus,gold_bbm_date,gold_bbm_locality,mo_sci_name,mo_genus,mo_date,mo_locality
0,191206,F19454,genus_mismatch_blocks_match,Gold BBM genus and MO genus differ or one is m...,lacera,lacera,2007-10-29,"victoria, saanich, observatory hill",inocybe lacera,inocybe,2007-10-29,"observatory hill, victoria, british columbia, ..."
1,166513,F19584,name_below_rule_threshold,Best gold name similarity is 0.654,cortinarius subsertipes,cortinarius,2008-11-05,"victoria, saanich, observatory hill",cortinarius decipientoides,cortinarius,2008-11-05,"observatory hill, victoria, british columbia, ..."
2,210193,F19553,name_below_rule_threshold,Best gold name similarity is 0.474,inocybe glabrodisca,inocybe,2008-10-26,"victoria, saanich, observatory hill",inocybe ceskae,inocybe,2008-10-26,"observatory hill, victoria, british columbia, ..."
3,209961,F19266,name_below_rule_threshold,Best gold name similarity is 0.5,inocybe mixtilis,inocybe,2004-12-11,"victoria, saanich, observatory hill",inocybe ceskae,inocybe,2004-12-11,"observatory hill, victoria, british columbia, ..."
4,202610,F19645,name_below_rule_threshold,Best gold name similarity is 0.5,inocybe soluta,inocybe,2009-05-14,"victoria, saanich, observatory hill",inocybe ericetorum,inocybe,2009-05-14,"observatory hill, victoria, british columbia, ..."
5,191211,F19450,genus_mismatch_blocks_match,Gold BBM genus and MO genus differ or one is m...,flocculosa,flocculosa,2007-10-29,"victoria, saanich, observatory hill",inocybe flocculosa,inocybe,2007-10-29,"observatory hill, victoria, british columbia, ..."
6,166825,F19593,name_below_rule_threshold,Best gold name similarity is 0.667,cortinarius californicus,cortinarius,2008-11-21,"victoria, saanich, observatory hill",cortinarius cinnabarinus,cortinarius,2008-11-21,"observatory hill, victoria, british columbia, ..."
7,132649,F16319,name_below_rule_threshold,Best gold name similarity is 0.65,cortinarius sertipes,cortinarius,2008-02-28,"behind the smaller dome, observatory hill, saa...",cortinarius miwok,cortinarius,2008-02-28,"observatory hill, victoria, british columbia, ..."
8,166061,F29822,genus_mismatch_blocks_match,Gold BBM genus and MO genus differ or one is m...,boletus truncatus,boletus,2008-10-17,"victoria, saanich peninsula, observatory hill;...",xerocomellus diffractus,xerocomellus,2008-10-17,"observatory hill, victoria, british columbia, ..."
9,72889,F19522,name_below_rule_threshold,Best gold name similarity is 0.579,inocybe praetervisa,inocybe,2008-07-03,"victoria, saanich, observatory hill",inocybe calida,inocybe,2008-07-03,"observatory hill, victoria, british columbia, ..."


## 10. DAP implementation / harmonization decay audit (category 07)

DAP is not only validation ground truth. It also records requested harmonization actions from 2025. This section compares those requested actions against the current local extracts to ask whether the prior work appears implemented now.

This is intentionally conservative: actions that require MO comments or GenBank comments are marked `cannot_assess_current_extract` because those fields are not preserved in the normalized CSVs. Re-fetch first if the live systems may have changed; this audit evaluates the current local data files.


In [ ]:
# DAP implementation/decay audit. OFFLINE — compares DAP requested actions to current local extracts.
import importlib
import pandas as pd
import dap_implementation_audit as DIA; importlib.reload(DIA)

dap_actions = DIA.build_audit()
DIA.write_csv(DIA.OUT_PATH, dap_actions)
DIA.write_csv(DIA.SUMMARY_PATH, DIA.summarize(dap_actions))

dap_impl = pd.DataFrame(dap_actions)
dap_summary = pd.DataFrame(DIA.summarize(dap_actions))
print(f"DAP action rows: {len(dap_impl)}")
display(dap_summary)
display(pd.crosstab(dap_impl["action_scope"], dap_impl["status"]))
display(dap_impl[dap_impl["status"] != "implemented"].head(30))


## 11. GenBank linkage — unlinked genomic data (category 01)

The paper (§5.1.2) singles out GenBank: ITS sequences derived from UBC vouchers
"frequently remain unlinked to either the Mushroom Observer or UBC lineage,"

 `data/genbank_ground_truth.csv` — 213 accessions from
the 2025 DAP sheet, each mapped to its UBC `F#` — is the ground truth; `genbank_audit.py`
checks whether the UBC record actually carries the accession.

**Result: 212 / 213 (99.5%) are unlinked** — only one UBC record cites its own GenBank
sequence, though all 213 vouchers exist in the collection.

In [17]:
import importlib, logging
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
import genbank_audit as GB; importlib.reload(GB)
GB.main()

GenBank linkage audit (collection ITS sequences vs UBC)
  collection accessions (DAP ground truth) : 213
  voucher F# present in BBM extract         : 213
  UBC -> GenBank (UBC record cites the acc.): 1
  UNLINKED on UBC lineage (category 01)     : 212  (99.5%)
  --- with fetched genbank_records.csv ---
  accessions actually fetched              : 213 / 213
  GenBank -> UBC (voucher cites the F#)     : 183
  breakdown 01 (missing identifier cross-references): 212
Saved per-accession -> /Users/wfrankel/Desktop/breakdowns_DES/reports/genbank_linkage.csv


## 12. Unified specimen lineage report — cross-platform digital fingerprint

This section builds `reports/specimen_lineage_report.csv`, a single action-ledger table over the current audit outputs. It does **not** perform new matching. Instead, it joins the latest BBM rows to explicit MO cross-references, MO resolver candidates, MyCoPortal and GBIF GUID coverage, and GenBank linkage.

Read this as the cross-platform tracing artifact: one row per BBM specimen, plus orphan rows for harvested-platform records whose GUID does not join back to the current BBM extract. It is the clearest bridge between the paper's "lineage tracing" method and the repository's automated full-collection audit.


In [ ]:
# Unified lineage report. OFFLINE — joins existing audit outputs, no network calls.
import importlib
import pandas as pd
import lineage_report as LR; importlib.reload(LR)

rows = LR.build_report()
lineage_path = LR.write_report(rows)
lineage = pd.DataFrame(rows)

bbm_lineage = lineage[lineage["specimen_key"].str.startswith(("BBM:", "BBM_ID:"))]
orphan_lineage = lineage[lineage["specimen_key"].str.startswith("ORPHAN:")]

print(f"lineage report: {lineage_path}")
print(f"BBM specimen rows: {len(bbm_lineage)}")
print(f"harvested-platform orphan rows: {len(orphan_lineage)}")
print(f"rows needing action: {(lineage['recommended_action'] != 'none').sum()}")

display(pd.DataFrame({
    "mo_explicit_link_status": bbm_lineage["mo_explicit_link_status"].value_counts(),
}).fillna(0).astype(int).reset_index(names="status"))

display(pd.DataFrame({
    "mycoportal_status": lineage["mycoportal_status"].value_counts(),
    "gbif_status": lineage["gbif_status"].value_counts(),
}).fillna(0).astype(int).reset_index(names="status"))

action_cols = [
    "specimen_key", "bbm_taxon", "mo_explicit_link_status",
    "mo_resolution_records", "mycoportal_status", "gbif_status",
    "genbank_accessions", "breakdown_categories", "recommended_action",
]
display(lineage[lineage["recommended_action"] != "none"][action_cols].head(30))


### Lineage spot check

The unified lineage report is a join over multiple outputs. This deterministic spot check samples representative cases and verifies that status, breakdown category, and recommended action agree. It is a reporting consistency check, not a curator review.


In [ ]:
# Deterministic spot check for lineage-report consistency.
import importlib
import csv
import pandas as pd
import spot_check_lineage as SCL; importlib.reload(SCL)

spot_rows = SCL.check_rows(lineage.to_dict("records") if "lineage" in globals() else pd.read_csv(LR.REPORTS_DIR / "specimen_lineage_report.csv").to_dict("records"))
SCL.Path(SCL.OUT_PATH).parent.mkdir(exist_ok=True)
with open(SCL.OUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(spot_rows[0].keys()))
    writer.writeheader(); writer.writerows(spot_rows)
spot = pd.DataFrame(spot_rows)
display(spot)
print(f"spot checks passed: {(spot['result'] == 'pass').sum()} / {len(spot)}")


## Methods & caveats

- **Most recent generated outputs are the current numbers.** Re-run fetches and then re-run the notebook / `run_audit.py` whenever BBM, MO, MyCoPortal, GBIF, or GenBank may have changed.
- **Goal 1 counts explicit links.** The representation table counts identifier cross-references in the current data. Resolver quadrants are separate candidate-pair outputs and should be used for Goal 2 / review queues.
- **BBM data**: full `collectionobject` table + joins to determination→taxon, collector→agent, collecting-event→locality (`get_bbm_records.py`).
- **MO reference formats caught**: `MO # 82752`, `MUOB 12345`, `Mushroom Observer observation #…`, mushroomobserver.org URLs. `MO posted as …` (no number) is a link with no id and is not looked up.
- **Coupling dictates method**: harvested-downstream platforms (MyCoPortal, GBIF) match by our GUID; independent platforms (MO, GenBank) match by the id we stored + attribute resolution.
- **Resolution**: rule-based predicates use name + exact date + locality/collector token overlap. The optional LLM tier only receives bounded candidate sets, is guardrailed after the model, and reports review/audit metadata.
- **Duplicate records**: `mo_duplicates.csv` and duplicate counts are review candidates, not confirmed duplicate findings. Current validation ground truth covers Observatory Hill / Ceska MO→UBC links only, not duplicate/non-duplicate labels.
- **Name drift**: current matching flags name mismatches but does not yet expand names through a synonym/accepted-name index. The MDS synonym API pipeline is the intended source for this layer.
- **Unified lineage report**: `specimen_lineage_report.csv` joins existing outputs into one cross-platform trace per BBM specimen, plus harvested-platform orphan rows. It is a reporting/action table, not a new matcher.
